# Bank Marketing — Exploratory Data Analysis

**Business question:** Can we predict whether a client will subscribe to a term deposit,
based on demographic, financial, and previous-campaign details?

**Dataset:** UCI Bank Marketing (id=222) — 45 211 client records from a Portuguese retail bank.

---
This notebook explores the dataset structure, class balance, feature distributions,
and relationships between predictors and the target variable.

Run from the repo root:
```bash
uv run jupyter lab experiments/notebooks/01_eda.ipynb
```

In [1]:
import sys
sys.path.insert(0, '../..')   # make src/ importable

import warnings
warnings.filterwarnings('ignore')

import matplotlib
matplotlib.rcParams['figure.dpi'] = 120

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

from src.config import settings
from src.data.ingest import fetch_raw_data, save_raw_to_disk
from src.data.preprocess import clean_raw
from src.visualization import plots

print('Setup complete.')

Setup complete.


## 1 — Load Dataset

In [2]:
import os
raw_csv = settings.data_raw_path / 'bank_marketing_raw.csv'

if raw_csv.exists():
    raw_df = pd.read_csv(raw_csv)
    print(f'Loaded from disk: {raw_csv}')
else:
    print('Downloading from UCI...')
    X, y = fetch_raw_data()
    save_raw_to_disk(X, y)
    raw_df = X.copy()
    raw_df['subscribed'] = y.values

print(f'Shape: {raw_df.shape}')
raw_df.head()

Loaded from disk: C:\Users\grege\Documents\My Docs\A_UofT\team_project_2\Bank-Marketing_ML_group3\data\raw\bank_marketing_raw.csv
Shape: (45211, 17)


,age,job,marital,education,default,balance,housing,loan,contact,day_of_week,month,duration,campaign,pdays,previous,poutcome,subscribed
0,58,management,married,tertiary,no,2143,yes,no,NaN,5,may,261,1,-1,0,NaN,no
1,44,technician,single,secondary,no,29,yes,no,NaN,5,may,151,1,-1,0,NaN,no
2,33,entrepreneur,married,secondary,no,2,yes,yes,NaN,5,may,76,1,-1,0,NaN,no
3,47,blue-collar,married,NaN,no,1506,yes,no,NaN,5,may,92,1,-1,0,NaN,no
4,33,NaN,single,NaN,no,1,no,no,NaN,5,may,198,1,-1,0,NaN,no


## 2 — Dataset Overview

In [3]:
raw_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 45211 entries, 0 to 45210
Data columns (total 17 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   age          45211 non-null  int64 
 1   job          44923 non-null  object
 2   marital      45211 non-null  object
 3   education    43354 non-null  object
 4   default      45211 non-null  object
 5   balance      45211 non-null  int64 
 6   housing      45211 non-null  object
 7   loan         45211 non-null  object
 8   contact      32191 non-null  object
 9   day_of_week  45211 non-null  int64 
 10  month        45211 non-null  object
 11  duration     45211 non-null  int64 
 12  campaign     45211 non-null  int64 
 13  pdays        45211 non-null  int64 
 14  previous     45211 non-null  int64 
 15  poutcome     8252 non-null   object
 16  subscribed   45211 non-null  object
dtypes: int64(7), object(10)
memory usage: 5.9+ MB


In [4]:
raw_df.describe(include='all').T

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
age,45211.0,NaN,NaN,NaN,40.93621,10.618762,18.0,33.0,39.0,48.0,95.0
job,44923,11,blue-collar,9732,NaN,NaN,NaN,NaN,NaN,NaN,NaN
marital,45211,3,married,27214,NaN,NaN,NaN,NaN,NaN,NaN,NaN
education,43354,3,secondary,23202,NaN,NaN,NaN,NaN,NaN,NaN,NaN
default,45211,2,no,44396,NaN,NaN,NaN,NaN,NaN,NaN,NaN
balance,45211.0,NaN,NaN,NaN,1362.272058,3044.765829,-8019.0,72.0,448.0,1428.0,102127.0
housing,45211,2,yes,25130,NaN,NaN,NaN,NaN,NaN,NaN,NaN
loan,45211,2,no,37967,NaN,NaN,NaN,NaN,NaN,NaN,NaN
contact,32191,2,cellular,29285,NaN,NaN,NaN,NaN,NaN,NaN,NaN
day_of_week,45211.0,NaN,NaN,NaN,15.806419,8.322476,1.0,8.0,16.0,21.0,31.0


## 3 — Target Variable (Class Imbalance)

In [5]:
eda_df = clean_raw(raw_df)
print('Subscription rate: {:.2%}'.format(eda_df['subscribed'].mean()))
eda_df['subscribed'].value_counts()

Subscription rate: 11.70%


subscribed
0    39922
1     5289
Name: count, dtype: int64

In [6]:
fig = plots.plot_target_distribution(eda_df)
plt.show()

> **Key insight:** Only ~11.7 % of clients subscribed. This significant class imbalance
> means accuracy alone is misleading — we focus on **ROC-AUC** and **F1** as primary metrics.

## 4 — Numeric Feature Distributions

In [7]:
fig = plots.plot_numeric_distributions(eda_df)
plt.show()

## 5 — Subscription Rate by Categorical Feature

In [8]:
fig = plots.plot_categorical_subscription_rate(eda_df)
plt.show()

## 6 — Correlation Heatmap

In [9]:
fig = plots.plot_correlation_heatmap(eda_df)
plt.show()

## 7 — Age by Subscription Outcome

In [10]:
fig = plots.plot_age_by_subscription(eda_df)
plt.show()

## 8 — Missing Value Analysis

The dataset uses `'unknown'` as a category placeholder rather than NaN.
After converting `'unknown'` → `NaN` in the cleaning step:

In [11]:
missing = eda_df.isnull().sum()
missing_pct = (missing / len(eda_df) * 100).round(2)
pd.DataFrame({'missing_count': missing, 'missing_%': missing_pct}).query('missing_count > 0')

,missing_count,missing_%
job,288,0.64
education,1857,4.11
contact,13020,28.80
poutcome,36959,81.75


## 9 — Key EDA Takeaways

| Finding | Implication |
|---------|-------------|
| 11.7 % positive class | Use ROC-AUC + F1; apply `class_weight='balanced'` |
| `pdays=999` means never contacted | Create a `previously_contacted` binary flag |
| `duration` only known post-call | **Exclude** from realistic production model |
| `poutcome='success'` → very high sub rate | Strong predictor |
| High `euribor3m` → lower subscription | Macro-economic context matters |
| `education` and `job` show variance | Include as encoded features |
